# NB-03: Section Structure & Numbering Validator

Parses the full section hierarchy, compares against the expected numbering from the Excel content inventory, audits subsection label naming, and counts equation environments.

## Step 1 — Parse full section tree

In [ ]:
import re, pathlib
from pathlib import Path

TEX_FILE = "jmlr-hypatiax-paper-final.tex"
source = Path(TEX_FILE).read_text(encoding="utf-8")
lines  = source.splitlines()

SECT_RE = re.compile(
    r'\\((?:sub)*section)\*?\{([^}]+)\}.*?(?:\\label\{([^}]+)\})?',
    re.M)

sections = []
for m in SECT_RE.finditer(source):
    level = m.group(1).count("sub")  # 0=section, 1=sub, 2=subsub
    title = m.group(2).strip()
    label = m.group(3) or ""
    lineno = source[:m.start()].count("\n") + 1
    sections.append((level, title, label, lineno))

indent = {0: "", 1: "  ", 2: "    "}
for level, title, label, lineno in sections:
    lbl_str = f"  [\\label{{{label}}}]" if label else ""
    print(f"L{level} {indent[level]}{title}{lbl_str}  (line {lineno})")

## Step 2 — Expected vs. actual section structure

In [ ]:
# Expected section numbering from Excel content inventory
# Format: (expected_number, title_fragment, expected_label)
EXPECTED = [
    (1, "Introduction",                          "sec:intro"),
    (2, "Related Work",                          "sec:related"),
    (3, "Empirical Evidence",                    "sec:llm_limitations"),
    (4, "Theoretical Framework",                 "sec:theory"),
    (5, "Problem Formulation",                   "sec:problem"),
    (6, "Benchmark Design",                      "sec:benchmark"),
    (7, "Methodology",                           "sec:method"),
    (8, "HypatiaX Architecture",                 "sec:architecture"),
    (9, "Experimental Setup",                    "sec:setup"),
    (10, "Results",                              "sec:results"),
    (11, "Discussion",                           "sec:discussion"),
    (12, "Conclusion",                           "sec:conclusion"),
]

print("Expected vs. Actual section structure (top-level only):")
print("-" * 80)
actual_sections = [(t, la, ln) for lv, t, la, ln in sections if lv == 0
                   and not t.startswith("*")]

for i, (exp_num, exp_frag, exp_label) in enumerate(EXPECTED):
    if i < len(actual_sections):
        act_title, act_label, act_line = actual_sections[i]
        title_ok  = exp_frag.lower() in act_title.lower()
        label_ok  = exp_label == act_label
        status_t  = "OK" if title_ok  else "MISMATCH"
        status_l  = "OK" if label_ok  else "MISMATCH"
        print(f"  §{exp_num:2d}  Title:  [{status_t}]  expected '{exp_frag}'  got '{act_title[:40]}'")
        print(f"        Label:  [{status_l}]  expected '{exp_label}'  got '{act_label}'")
    else:
        print(f"  §{exp_num:2d}  MISSING section")

## Step 3 — Subsection label audit

In [ ]:
# Check subsection labels follow section numbering pattern
# e.g., subsections of §7 should have labels containing '7' or relevant names
print("Subsection label audit (sections 7-10):")
print("-" * 80)
relevant = [(lv, t, la, ln) for lv, t, la, ln in sections
            if lv == 1 and ln >= 0]

for level, title, label, lineno in relevant:
    if label:
        print(f"  {title[:50]:<52}  \\label{{{label}}}  (line {lineno})")
    else:
        print(f"  {title[:50]:<52}  [NO LABEL]  (line {lineno})")

## Step 4 — Equation environment inventory

In [ ]:
# Count equations and check numbering environment types
EQ_ENVS  = re.findall(r'\\begin\{(equation|align|gather|multline)\*?\}', source)
LABEL_EQ = re.findall(r'\\begin\{equation\}.*?\\label\{(eq:[^}]+)\}', source, re.DOTALL)

print(f"Numbered equation environments: {len(EQ_ENVS)}")
print(f"  equation : {EQ_ENVS.count('equation')}")
print(f"  align    : {EQ_ENVS.count('align')}")
print(f"  gather   : {EQ_ENVS.count('gather')}")
print(f"  multline : {EQ_ENVS.count('multline')}")
print()
print(f"Equation labels (eq:*): {len(LABEL_EQ)}")
for k in LABEL_EQ:
    refs_in_text = source.count(f"\\ref{{{k}}}") + source.count(f"\\eqref{{{k}}}")
    print(f"  {k:<30}  referenced {refs_in_text}x")